# This notebook runs the quantitative analysis
Please provide the right configuration and run the notebook through to the end

In [12]:
import logging
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torchaudio
import yaml

from functools import partial
from logging.handlers import RotatingFileHandler
from IPython.display import Audio
from jsonschema import validate, ValidationError
from scipy.io.wavfile import write
from torchmetrics.classification import MulticlassAccuracy

from esc_audio_dataset import ESCAudioDataset, LABEL_MAP, REVERSE_LABEL_MAP
from bcos.modules.bcosconv2d import BcosConv2d
from config.config_validation_template import CONFIG_TEMPLATE
from custom_logger_formatter import CustomLoggerFormatter
from main import DATASET_MAPPING
from resnet_bcos import make_resnet18, make_resnet34, make_resnet50
from resnet_18_baseline import BaselineResNet
from tune import TUNABLE_PARAMS
from augment import AudioAugment, SpecAugment
from data import to_dataloaders

import quant_analysis_utils as q_utils


### 0. Config & setup

In [2]:
EXPERIMENT_DIR = os.path.join("output", "12-06-2026--22-39") # change directory to the specific experiment

BEST_MODEL_DIR = os.path.join("job_0", "best_model") # folder containing .pt model, relative from `EXPERIMENT_DIR`

DEVICE = "cuda" # use if CUDA or ROCm

In [3]:
# Initialise Logger.
def _make_stream_handler(level: int) -> logging.StreamHandler:
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(level)
    ch.setFormatter(CustomLoggerFormatter())
    return ch

level: int=logging.DEBUG
logger = logging.getLogger("test logger")
logger.setLevel(level)
logger.propagate = False  
ch = _make_stream_handler(level)
logger.addHandler(ch)
f_ch = RotatingFileHandler(f"{EXPERIMENT_DIR}/quantitative_analysis.log")
f_ch.setLevel(level)
f_ch.setFormatter(
    logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
    )
)
logger.addHandler(f_ch)

logger.info(f"This file logs the testing process.")

2026-06-13 02:31:38,852 - test logger - INFO - This file logs the testing process. (1909239882.py:23)


In [ ]:
# validate the provided config file.
# with open(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/run_config.yml", 'r') as stream:
with open(os.path.join(EXPERIMENT_DIR, BEST_MODEL_DIR, "run_config.yml"), 'r') as stream:
    CONFIG = yaml.safe_load(stream)

# Get the general config settings from the main yaml
with open(os.path.join(EXPERIMENT_DIR, "config.yml"), 'r') as stream:
    general_config = yaml.safe_load(stream)

CONFIG["general"] = general_config["general"]

try:
    validate(general_config, CONFIG_TEMPLATE)
except ValidationError as e:
    raise ValidationError(
        "\x1b[31;1mA validation error occurred in the config file" \
        f": {e.message}\x1b[0m"
    ) from e

for tunable_param in TUNABLE_PARAMS.keys():
    if type(CONFIG[tunable_param]) == list:
        CONFIG[tunable_param] = CONFIG[tunable_param][0]

logger.info("config: %s", CONFIG)

In [7]:
with open(os.path.join(EXPERIMENT_DIR, BEST_MODEL_DIR, "run_config.yml"), 'r') as stream:
    CONFIG = yaml.safe_load(stream)

CONFIG = CONFIG["jobs"]["job0"]

for k in CONFIG.keys():
    if type(CONFIG[k]) == list:
        CONFIG[k] = CONFIG[k][0]

### 1. Load data


In [8]:
dataset = ESCAudioDataset(
    data_dirs=DATASET_MAPPING["dirs"],
    folds=DATASET_MAPPING["train_folds"],
    csv_path=DATASET_MAPPING["csv_path"],
    target_sr=CONFIG["sample_rate"],
    duration=CONFIG["duration"],
    n_fft=CONFIG["n_fft"],
    hop_length=CONFIG["hop_length"],
    n_mels=CONFIG["n_mels"],
    top_db=CONFIG["top_db"],
    pre_transform=AudioAugment(
        sample_rate=CONFIG["sample_rate"],
    ),
    post_transform=SpecAugment()
)
logger.debug(f"Dataset size: {len(dataset)}")
logger.debug(f"Shape of first x element: {dataset[0][0].shape}")
logger.debug(f"First y element: {dataset[0][1]}")
test_data = ESCAudioDataset(
    data_dirs=DATASET_MAPPING["dirs"],
    folds=DATASET_MAPPING["test_folds"],
    csv_path=DATASET_MAPPING["csv_path"],
    target_sr=CONFIG["sample_rate"],
    duration=CONFIG["duration"],
    n_fft=CONFIG["n_fft"],
    hop_length=CONFIG["hop_length"],
    n_mels=CONFIG["n_mels"],
    top_db=CONFIG["top_db"],
)
logger.debug(f"Test dataset size: {len(test_data)}")

# Normalise
logger.debug("Fitting normalisation.")
dataset.fit_normalisation(list(range(len(dataset))))
test_data.mean = dataset.mean
test_data.std = dataset.std
logger.debug(
    "Normalisation fitted: "
    f"mean={dataset.mean}, std={dataset.std}"
)

2026-06-13 02:33:33,160 - test logger - DEBUG - Dataset size: 320 (622359652.py:16)
2026-06-13 02:33:33,167 - test logger - DEBUG - Shape of first x element: torch.Size([1, 128, 216]) (622359652.py:17)
2026-06-13 02:33:33,171 - test logger - DEBUG - First y element: 0 (622359652.py:18)
2026-06-13 02:33:33,187 - test logger - DEBUG - Test dataset size: 80 (622359652.py:30)
2026-06-13 02:33:33,188 - test logger - DEBUG - Fitting normalisation. (622359652.py:33)
2026-06-13 02:33:34,299 - test logger - DEBUG - Normalisation fitted: mean=-7.934070110321045, std=15.656573295593262 (622359652.py:37)


### 2. Load model

In [9]:
models = {
    "resnet18_bcos": (
        make_resnet18, {
            "logger": logger,
            "num_classes": dataset.get_n_classes(),
            "in_chans" : 1,
            "small_inputs": CONFIG["model_params"].get("small_inputs", True),
            "conv_layer": partial(BcosConv2d, b=CONFIG["b"], max_out=2),
        }
    ),
    "resnet34_bcos": (
        make_resnet34, {
            "logger": logger,
            "num_classes": dataset.get_n_classes(),
            "in_chans" : 1,
            "small_inputs": CONFIG["model_params"].get("small_inputs", True),
            "conv_layer": partial(BcosConv2d, b=CONFIG["b"], max_out=2),
        }
    ),
    "resnet50_bcos": (
        make_resnet50, {
            "logger": logger,
            "num_classes": dataset.get_n_classes(),
            "in_chans" : 1,
            "small_inputs": CONFIG["model_params"].get("small_inputs", True),
            "conv_layer": partial(BcosConv2d, b=CONFIG["b"], max_out=2),
        }
    ),
    "resnet18_baseline": (
        lambda **kwargs: BaselineResNet(kwargs["num_classes"]),
        {"logger": logger, "num_classes": dataset.get_n_classes()}
    )   
}
model = None
for name, (cls, kwargs) in models.items():
    if CONFIG['model'].lower() in name:
        model = cls(**kwargs)
        break
assert model is not None, \
    f"Provided model in config does not exist ({model})."

logger.debug(f"Model:\n{model}")
logger.debug("Total number of parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)

model = model.to(DEVICE)


2026-06-13 02:33:37,986 - test logger - DEBUG - Model:
EnhancedResNet(
  (conv1): BcosConv2d(
    B=2.608695531936324, max_out=2,
    (linear): NormedConv2d(1, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  )
  (bn1): DetachablePositionNorm2dNoBias((64,), eps=1e-05, elementwise_affine=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): BcosConv2d(
        B=2.608695531936324, max_out=2,
        (linear): NormedConv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (bn1): DetachablePositionNorm2dNoBias((64,), eps=1e-05, elementwise_affine=True)
      (conv2): BcosConv2d(
        B=2.608695531936324, max_out=2,
        (linear): NormedConv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (bn2): DetachablePositionNorm2dNoBias((64,), eps=1e-05, elementwise_affine=True)
    )
    (1): BasicBlock(
      (conv1): BcosConv2d(
        B=2.608695531936324, max_out=2,
        (linear)

In [10]:
model_file_path = [entry for entry in os.listdir(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}") if entry.endswith(".pth")][0]

state_dict = torch.load(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/{model_file_path}", weights_only=True)
model.load_state_dict(state_dict)

<All keys matched successfully>

### 2.5 Get model accuracy

In [20]:
metric = MulticlassAccuracy(num_classes=10).to(DEVICE)

test_dataloader = to_dataloaders(
        [test_data], 
        batch_sizes=[CONFIG["batch_size"]], 
        shuffles=[False],
        logger=logger,
        num_workers=0,
        pin_memory=False,
    )[0]

with torch.no_grad():
    for inputs, labels in test_dataloader:
        outputs = model(inputs.to(DEVICE))
        metric.update(outputs, labels.to(DEVICE))

accuracy = metric.compute()
print(f"Accuracy on test set: {accuracy.item():.2f}")

2026-06-13 02:40:54,244 - test logger - DEBUG - Converting dataset of 80 elements into DataLoader with 80 partitions of size 1. (data.py:42)
Accuracy on test set: 0.90


### 3. Grid pointing game

In [ ]:
# train
train_grid_dir = f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/grid_train/"
os.makedirs(train_grid_dir, exist_ok=True)
train_grid_scores_absolute, train_grid_scores_weighted = q_utils.grid_pointing_game(
    dataset, 
    model, 
    DEVICE, 
    logger, 
    first_img_output_dir=train_grid_dir
)
logger.critical(f"Absolute grid pointing game results on train set: Total pairs evaluated: {len(train_grid_scores_absolute)} | Mean score: {train_grid_scores_absolute.mean() * 100:.5f}% | std: {train_grid_scores_absolute.std() * 100:.5f}%")
logger.critical(f"Relative grid pointing game results on train set: Total pairs evaluated: {len(train_grid_scores_weighted)} | Mean score: {train_grid_scores_weighted.mean() * 100:.5f}% | std: {train_grid_scores_weighted.std() * 100:.5f}%")

# test
test_grid_dir = f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/grid_test/"
os.makedirs(test_grid_dir, exist_ok=True)
test_grid_scores_absolute, test_grid_scores_weighted = q_utils.grid_pointing_game(
    test_data, 
    model, 
    DEVICE, 
    logger, 
    first_img_output_dir=test_grid_dir
)
logger.critical(f"Grid pointing game results on test set: Total pairs evaluated: {len(test_grid_scores_absolute)} | Mean score: {test_grid_scores_absolute.mean() * 100:.5f}% | std: {test_grid_scores_absolute.std() * 100:.5f}%")
logger.critical(f"Grid pointing game results on test set: Total pairs evaluated: {len(test_grid_scores_weighted)} | Mean score: {test_grid_scores_weighted.mean() * 100:.5f}% | std: {test_grid_scores_weighted.std() * 100:.5f}%")

torch.cuda.empty_cache()

### 4. Positive/negative contribution mask accuracy
see how well the model predicts when only being able to use the pixels it used positively

In [ ]:
# train
train_mask_dir = f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/mask_train/"
os.makedirs(train_mask_dir, exist_ok=True)
original_train_accuracy, positive_masked_train_accuracy, negative_masked_train_accuracy = q_utils.positive_contribution_masking_accuracy(
    dataset=dataset,
    model=model,
    DEVICE=DEVICE,
    logger=logger,
    first_img_output_dir=train_mask_dir
)

logger.info(f"Original train accuracy: {original_train_accuracy*100:.5f}")
logger.info(f"Positive masked train accuracy: {positive_masked_train_accuracy*100:.5f}")
logger.info(f"Negative masked train accuracy: {negative_masked_train_accuracy*100:.5f}")

# test
test_mask_dir = f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/mask_test/"
os.makedirs(test_mask_dir, exist_ok=True)
original_test_accuracy, positive_masked_test_accuracy, negative_masked_test_accuracy = q_utils.positive_contribution_masking_accuracy(
    dataset=test_data,
    model=model,
    DEVICE=DEVICE,
    logger=logger,
    first_img_output_dir=test_mask_dir
)

logger.info(f"Original test accuracy: {original_test_accuracy*100:.5f}")
logger.info(f"Positive masked test accuracy: {positive_masked_test_accuracy*100:.5f}")
logger.info(f"Negative masked test accuracy: {negative_masked_test_accuracy*100:.5f}")

torch.cuda.empty_cache()